In [4]:
import pandas as pd


# Load data
data = pd.read_csv("data.csv")
# Concatenate Job Title and Description into one column
data['job_with_description'] = data['Job Title'] + '  ' + data['Description']
data['job_with_description']



0        développeur full stack java  flutter confirmé ...
1        consultant devops sénior  kubernetes grafana  ...
2        gestionnaire dapplication sénior  servicenow j...
3        data scientist confirmé  servicenow bigquery  ...
4        développeur full stack java  vuejs gcp sénior ...
                               ...                        
16463    product owner scurit api junior  le centre d'e...
16464    dveloppeur back javamobile  basée à lille, lyo...
16465    amoa souscription et gestion iard hors sinistr...
16466    dveloppeur full stack  basée à lille, lyon, na...
16467    ingnieure iamiga lille 3j par semaine  en quel...
Name: job_with_description, Length: 16468, dtype: object

In [6]:
import time
import numpy as np
from sentence_transformers import SentenceTransformer
import faiss
import torch

# Load data
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = SentenceTransformer('all-MiniLM-L6-v2')
sentences = data['job_with_description'].dropna()

sentence_vectors = model.encode(sentences,show_progress_bar=True).astype('float32')

Batches:   0%|          | 0/515 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
sentence_vectors = sentence_vectors / np.linalg.norm(sentence_vectors, axis=1, keepdims=True)

# Hardcoded queries
query_texts = [
    "A highly skilled Full Stack Developer with 3+ years of experience in building scalable web applications using React, Node.js, and MongoDB. Proficient in RESTful API design, DevOps basics, and CI/CD pipelines. Passionate about clean code, responsive design, and solving real-world problems with elegant web solutions.",
    "Experienced Data Scientist with a strong background in Python, machine learning, and statistical analysis. Adept at building predictive models, performing data wrangling, and deriving actionable insights from large datasets. Skilled in using tools like Scikit-learn, TensorFlow, Pandas, and Jupyter notebooks.",
    "Dedicated Cybersecurity Analyst with expertise in network security, vulnerability assessments, and incident response. Familiar with SIEM tools, ethical hacking, and security best practices. Holds certifications like CompTIA Security+ and working toward CISSP. Strong commitment to protecting digital assets and ensuring compliance.",
    "DevOps Engineer with a passion for automation and infrastructure-as-code. Experienced in AWS, Docker, Kubernetes, and CI/CD using Jenkins and GitLab. Skilled in monitoring systems using Prometheus and Grafana, and ensuring system reliability and scalability through best DevOps practices.",
    "Creative and detail-oriented Mobile App Developer proficient in Flutter and React Native. Delivered multiple cross-platform apps with intuitive UI and robust backend integration. Familiar with Firebase, REST APIs, and modern state management solutions like Bloc and Redux."
]
query_vectors = model.encode(query_texts).astype('float32')
query_vectors = query_vectors / np.linalg.norm(query_vectors, axis=1, keepdims=True)

# FAISS Indexes
d = sentence_vectors.shape[1]
k = 3

flat_index = faiss.IndexFlatIP(d)
flat_index.add(sentence_vectors)

nlist = 50
m = 8
nbits = 8
quantizer = faiss.IndexFlatIP(d)
ivfpq_index = faiss.IndexIVFPQ(quantizer, d, nlist, m, nbits)
ivfpq_index.train(sentence_vectors)
ivfpq_index.add(sentence_vectors)
ivfpq_index.nprobe = 10

hnsw_index = faiss.IndexHNSWFlat(d, 32)
hnsw_index.hnsw.efSearch = 64
hnsw_index.hnsw.efConstruction = 200
hnsw_index.add(sentence_vectors)

# Metrics
def precision_at_k(true, pred, k): return len(set(true[:k]) & set(pred[:k])) / k
def recall_at_k(true, pred, k): return len(set(true[:k]) & set(pred[:k])) / k
def f1_at_k(true, pred, k):
    p = precision_at_k(true, pred, k)
    r = recall_at_k(true, pred, k)
    return 2 * p * r / (p + r) if p + r > 0 else 0.0
def average_precision_at_k(true, pred, k):
    hits, score = 0, 0.0
    true_set = set(true[:k])
    for i, idx in enumerate(pred[:k]):
        if idx in true_set:
            hits += 1
            score += hits / (i + 1)
    return score / min(len(true_set), k) if true_set else 0.0
def mean_reciprocal_rank(true, pred, k):
    true_set = set(true[:k])
    for i, idx in enumerate(pred[:k]):
        if idx in true_set:
            return 1 / (i + 1)
    return 0.0
def dcg_at_k(relevance, k): return sum((2 ** rel - 1) / np.log2(i + 2) for i, rel in enumerate(relevance[:k]))
def ndcg_at_k(true, pred, k):
    true_set = set(true[:k])
    relevance = [1 if idx in true_set else 0 for idx in pred[:k]]
    ideal_relevance = sorted(relevance, reverse=True)
    dcg = dcg_at_k(relevance, k)
    idcg = dcg_at_k(ideal_relevance, k)
    return dcg / idcg if idcg > 0 else 0.0

metrics = ["recall", "precision", "f1", "map", "mrr", "ndcg"]
results_ivf = {m: 0.0 for m in metrics}
results_hnsw = {m: 0.0 for m in metrics}

flat_times, ivf_times, hnsw_times = [], [], []

with open("recommendation_results.txt", "w", encoding="utf-8") as out:
    for idx, query_vector in enumerate(query_vectors):
        out.write(f"\n🔍 Query: {query_texts[idx]}\n")

        start = time.perf_counter()
        true_ids = flat_index.search(query_vector.reshape(1, -1), k)[1][0]
        flat_times.append(time.perf_counter() - start)

        start = time.perf_counter()
        ivf_ids = ivfpq_index.search(query_vector.reshape(1, -1), k)[1][0]
        ivf_times.append(time.perf_counter() - start)

        start = time.perf_counter()
        hnsw_ids = hnsw_index.search(query_vector.reshape(1, -1), k)[1][0]
        hnsw_times.append(time.perf_counter() - start)

        out.write("Flat Index:\n")
        for i in true_ids: out.write(f"- {sentences[i]}\n")

        out.write("IVFPQ Index:\n")
        for i in ivf_ids: out.write(f"- {sentences[i]}\n")

        out.write("HNSW Index:\n")
        for i in hnsw_ids: out.write(f"- {sentences[i]}\n")

        for metric, func in zip(metrics, [recall_at_k, precision_at_k, f1_at_k, average_precision_at_k, mean_reciprocal_rank, ndcg_at_k]):
            results_ivf[metric] += func(true_ids, ivf_ids, k)
            results_hnsw[metric] += func(true_ids, hnsw_ids, k)

    n = len(query_vectors)
    for m in metrics:
        results_ivf[m] /= n
        results_hnsw[m] /= n

    avg_flat_time = sum(flat_times) / n
    avg_ivf_time = sum(ivf_times) / n
    avg_hnsw_time = sum(hnsw_times) / n

    out.write("\n📈 Evaluation Summary on hardcoded queries\n")
    out.write(f"{'Metric':<10} | {'IVFPQ':>6} | {'HNSW':>6}\n")
    out.write("-" * 30 + "\n")
    for m in metrics:
        out.write(f"{m.upper():<10} | {results_ivf[m]:6.2f} | {results_hnsw[m]:6.2f}\n")

    out.write("\n⏱️ Average Search Time (seconds per query)\n")
    out.write(f"{'Flat':<10}: {avg_flat_time:.6f}\n")
    out.write(f"{'IVFPQ':<10}: {avg_ivf_time:.6f}\n")
    out.write(f"{'HNSW':<10}: {avg_hnsw_time:.6f}\n")
    # Avoid division by zero
    ivf_speedup = avg_flat_time / avg_ivf_time if avg_ivf_time > 0 else float('inf')
    hnsw_speedup = avg_flat_time / avg_hnsw_time if avg_hnsw_time > 0 else float('inf')

    out.write(f"\n⚡ Speedup Comparison:\n")
    out.write(f"IVFPQ Speedup      : {ivf_speedup:.2f}x\n")
    out.write(f"HNSW Speedup       : {hnsw_speedup:.2f}x\n")


In [9]:
import torch
print(torch.cuda.device_count())  # -> 2


0
